# 상습침수 지점 리포트 + 강우 임시 검증 (3단계 보정)

> 2026-06-19 · 라벨 신뢰도 작업 종합 · 공공관측소 10분 강우로 임시 검증

**목적** — 신뢰 도로침수 센서를 실제 강우와 교차검증해 진짜 상습침수 지점을 가리되, **강우가 절대 기준이 아님**을 반영.

**강우는 절대 기준이 아니다 (중요)** — 침수는 강우 외 원인으로도 발생: **상수도관 파열·하수 역류·하천 역류(상류 강우)·융설·공사·조위** 등. 또 관측소 ~1.5km라 **국소 호우를 놓칠 수 있음**. 따라서:
- **강우 동반(있음) = 강한 양성 증거** → 강우확인.
- **강우 없음 = 약한 음성 증거** → 시계열 형태로 갈라야: stuck/단일값/겨울지배면 **아티팩트**, 솟았다 회복하는 다양한 에피소드면 **비강우 실침수 가능 → 보류**(버리지 않음).

**3단계 신뢰등급**: 강우확인 / 부분강우 / **보류(비강우·원거리)** / 아티팩트(형태 비정상에 한정) / 보류(미검증=좌표결측).


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from pathlib import Path
OUT=Path("dataset/processed/eda_based"); FIG=Path("reports/figures_label"); GIS=Path("reports/gis")
korf=next((c for c in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR'] if c in {f.name for f in fm.fontManager.ttflist}),None)
if korf: plt.rcParams['font.family']=korf
plt.rcParams['axes.unicode_minus']=False
def hav(a,b,c,d):
    R=6371000.0;p1,p2=np.radians(a),np.radians(c);dp=np.radians(c-a);dl=np.radians(d-b)
    return 2*R*np.arcsin(np.sqrt(np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2))
trust=pd.read_parquet(OUT/"road_flood_sensor_trust.parquet")
audit=pd.read_parquet(OUT/"road_flood_sensor_audit.parquet")[['sensor_id','stuck_pos','max_ep','top1val','nuniq','winter']]
rn=pd.read_parquet("dataset/processed/cleaned/road_node.parquet",columns=['sensor_id','지점명','자치구','배수구역','lat','lon'])
rc=pd.read_parquet("dataset/processed/rain_station_coords.parquet",columns=['station_id','lat','lon'])
cand=trust[trust['판정_final'].isin(['진짜상습','미결'])].sensor_id.tolist()
rnn=rn[rn.sensor_id.isin(cand)].dropna(subset=['lat','lon']); m={}
for _,r in rnn.iterrows():
    d=hav(r.lat,r.lon,rc.lat.values,rc.lon.values); j=int(np.argmin(d)); m[r.sensor_id]=(rc.iloc[j].station_id,round(float(d[j])))
rain=pd.read_parquet(OUT/"rain_features_10min.parquet",columns=['station_id','timestamp','is_recent_rain_6h'])
RE=rain.timestamp.max(); rg={k:v.set_index('timestamp')['is_recent_rain_6h'] for k,v in rain.groupby('station_id')}
pan=pd.read_parquet(OUT/"road_panel_10min.parquet",columns=['sensor_id','ts10','road_max','flood_t6','month'])
def stat(sid):
    s=pan[pan.sensor_id==sid]; pos=s[s.flood_t6==1]
    if not len(pos): return None
    f=s.flood_t6.values; chg=np.r_[True,f[1:]!=f[:-1]]; ep=int(((chg)&(f==1)).sum())
    worst=pos.loc[pos.road_max.idxmax()]; ra=np.nan; dist=np.nan; cov=np.nan
    if sid in m:
        st=m[sid][0]; dist=m[sid][1]; fp=pos[pos.ts10<=RE]
        if len(fp) and st in rg:
            sub=rg[st].reindex(fp.ts10.values); cov=round(sub.notna().mean(),2)
            ra=round(sub.dropna().mean(),2) if sub.notna().any() else np.nan
    info=rn[rn.sensor_id==sid].iloc[0]
    return dict(sensor_id=sid,자치구=info['자치구'],배수구역=info['배수구역'],lat=info['lat'],lon=info['lon'],
        에피소드=ep,침수일수=pos.ts10.dt.normalize().nunique(),침수bin=len(pos),최대수위cm=int(pos.road_max.max()),
        여름비율=round(pos.month.isin([6,7,8,9]).mean(),2),최악일=str(worst.ts10)[:10],강우동반=ra,강우cov=cov,강우거리=dist)
R=pd.DataFrame([x for x in (stat(s) for s in cand) if x]).merge(audit,on='sensor_id',how='left')
def grade(r):
    abn=(r.stuck_pos>=0.5)or(r.max_ep>=144)or(r.top1val>=0.9)or(r.winter>=0.6)  # 시계열 비정상
    if pd.notna(r.강우동반):
        if r.강우동반>=0.6: return '강우확인'
        if r.강우동반>=0.3: return '부분강우'
        return '아티팩트' if abn else '보류(비강우/원거리)'
    return '아티팩트' if abn else '보류(미검증)'
R['신뢰등급']=R.apply(grade,axis=1)
R['판정_heur']=R.sensor_id.map(dict(zip(trust.sensor_id,trust['판정_final'])))
print("준비:",len(R),"센서 | 신뢰등급:",R['신뢰등급'].value_counts().to_dict())


## 1. 강우 임시 검증 + 형태 보정

In [ ]:
print("[시계열 '진짜상습' 10개 강우 재검증]")
h=R[R.판정_heur=='진짜상습'].sort_values('강우동반',ascending=False)
print(h[['sensor_id','자치구','에피소드','여름비율','최악일','강우cov','강우동반','신뢰등급']].to_string(index=False))
print("\n→ 갈현동·조원로·역삼동 등 무강우 = 형태는 정상(stuck/단일값 아님) → '아티팩트' 아닌 '보류(비강우/원거리)'. 강우확인만 고신뢰.")
print("\n[미결 22개]:",R[R.판정_heur=='미결']['신뢰등급'].value_counts().to_dict())
print("아티팩트로 확정된 후보:",int((R.신뢰등급=='아티팩트').sum()),"개 (형태 스크리닝을 이미 통과해 0에 가까움)")
fig,ax=plt.subplots(figsize=(11,4))
hh=R[R.판정_heur=='진짜상습'].dropna(subset=['강우동반']).sort_values('강우동반')
col={'강우확인':'seagreen','부분강우':'olive','보류(비강우/원거리)':'darkorange','보류(미검증)':'gray','아티팩트':'indianred'}
ax.barh(hh.sensor_id,hh.강우동반,color=[col[g] for g in hh.신뢰등급])
ax.axvline(0.6,color='k',ls='--',lw=.8); ax.axvline(0.3,color='gray',ls=':',lw=.8)
ax.set_xlabel('강우 동반율'); ax.set_title("'진짜상습' 10개 강우 검증 (초록=확인 / 주황=보류·비강우)")
plt.tight_layout(); plt.savefig(FIG/"03_rain_validation.png",dpi=110); plt.show()


## 2. 상습침수 지점 리포트

In [ ]:
conf=R[R.신뢰등급=='강우확인'].sort_values('에피소드',ascending=False)
hold=R[R.신뢰등급.str.startswith('보류')].sort_values('침수bin',ascending=False)
print("=== A. 강우확인 상습침수 %d지점 (고신뢰) ==="%len(conf))
print(conf[['sensor_id','자치구','배수구역','에피소드','침수일수','최대수위cm','여름비율','최악일','강우동반']].to_string(index=False))
print("\n=== B. 보류 (비강우/원거리·미검증) — 버리지 않음, 추가조사 대상 (상위) ===")
print(hold[['sensor_id','자치구','에피소드','침수bin','여름비율','최악일','강우동반','강우cov']].head(8).to_string(index=False))
print("  ※ 갈현동 466-9: 침수 1021bin·가을·무강우(cov0.57)로 가장 의심 크나, 형태 정상이라 아티팩트 확정은 보류.")
R.sort_values(['신뢰등급','에피소드'],ascending=[True,False]).to_parquet(OUT/"recurrent_flood_report.parquet",index=False)
fig,ax=plt.subplots(1,2,figsize=(14,5))
c=conf.sort_values('에피소드'); ax[0].barh(c.sensor_id+' ('+c.자치구.str.replace('청','')+')',c.에피소드,color='seagreen')
for i,(e,d) in enumerate(zip(c.에피소드,c.최대수위cm)): ax[0].text(e,i,f' {e}ep/{d}cm',va='center',fontsize=8)
ax[0].set_title('강우확인 상습침수: 에피소드/최대수위'); ax[0].set_xlabel('침수 에피소드')
for gname,gd in R.dropna(subset=['lat']).groupby('신뢰등급'):
    ax[1].scatter(gd.lon,gd.lat,c=col.get(gname,'gray'),label=gname,s=42,alpha=.85)
for _,r in conf.iterrows(): ax[1].annotate(r.sensor_id.split()[0],(r.lon,r.lat),fontsize=7)
ax[1].set_title('상습침수 후보 지도 (신뢰등급)'); ax[1].legend(fontsize=7); ax[1].set_xlabel('lon'); ax[1].set_ylabel('lat')
plt.tight_layout(); plt.savefig(FIG/"04_report_map.png",dpi=110); plt.show()
try:
    import geopandas as gpd; from shapely.geometry import Point
    g=R.dropna(subset=['lat','lon']).copy()
    gdf=gpd.GeoDataFrame(g.drop(columns=['lat','lon']),geometry=[Point(x) for x in zip(g.lon,g.lat)],crs='EPSG:4326')
    gdf.to_file(GIS/"recurrent_flood_report.gpkg",layer="flood_spots",driver="GPKG"); gdf.to_file(GIS/"recurrent_flood_report.geojson",driver="GeoJSON")
    print("QGIS 저장:",GIS/"recurrent_flood_report.gpkg")
except Exception as e: print("gpkg skip",e)


## 결론

- **강우는 절대 기준이 아님**(상수도·하수역류·하천역류·융설·공사 등 비강우 침수 + 관측소 ~1.5km 거리누락) → 강우 부재만으로 아티팩트라 단정하지 않는다.
- **강우 동반 = 강한 양성**: **강우확인 6지점**(시흥동 882-61·성산로 494-30·신영동 165·대림동 862-5·둔촌동 218-6·월계동 9-2)은 고신뢰 상습침수. 모두 여름집중·강우 0.75~1.0.
- **강우 부재 + 형태 정상 = 보류(비강우/원거리)**: 기존 '강우반증' 16개가 모두 여기로 완화(시계열 형태가 정상=stuck/단일값 아님). 즉 **아티팩트로 확정된 후보 0** — 강우만으론 못 버린다.
  - 단 **갈현동 466-9**(1021bin·가을·무강우)는 의심이 가장 크니 추가조사 1순위(하천역류/관파열/잔여아티팩트 구분 필요).
- **미검증 2**(하월곡동·사당2동, 좌표 결측)도 보류.

**의의** — 강우는 '확정 도구'가 아니라 **양성 확증 + 위험순위 도구**. 6지점은 확신, 나머지는 '버림'이 아니라 '보류(추가증거 대기)'로 두는 게 정직.

**다음 (추가 증거 시)** — 비강우 보류 센서에 **하천수위·상수도 사고·하수 역류 기록**을 교차해 비강우 실침수 vs 잔여 아티팩트 구분 / 정식 AWS로 거리누락 보완.
